# A3 移植 (8/8)：踩坑清单、遗留成本与下一步

> **一句话结论**：这批工作踩过的坑都按"症状→根因→修法"记录；剩余成本集中在 resolver AI-Core 化、int8 SFA 差距、量化 Q 路径与 prefill pack 融合。

## 1. 真实踩坑（入门版）

| 坑 | 根因 | 修法 |
| --- | --- | --- |
| GM 越界写（507035） | kernel 入口参数顺序与 op def 错位，静默换位指针 | 以 `*_def.cpp` 输入输出顺序逐位核对，顶部注释标注顺序 |
| 标量读到旧值/写错位置 | 标量读写底层用 `absAddr`，不是 `bufferAddr` | 标量用绝对下标 `GetValue/SetValue`，或走 TQue |
| sum/amax 脏读 | 向量流水线(V)与标量单元(S)不同步 | 取事件 id 做 `V_S`/`S_V` 栅栏；id 由 TPipe 取，不自选 |
| mapping 读到陈旧行号 | GM 标量直读在 A3 工具链不稳定 | 标量 `DataCopyEx` 进 UB 快照，寄存器数组内循环不再读标量 |
| TQue 对齐问题 | buffer 未 32B 对齐 | 按 32B 对齐分配 |
| 算子库缺符号 | OPP 装器共享库是累积产物，未全家族一次构建 | 算子私有 OPP 独立构建，全家族一次打包 |

## 2. 遗留结构性成本

- Resolver AICPU +5.24 ms/步：后续可 AI-Core 化；
- int8 SFA 比 bf16 SFA 慢 1.34×；
- 量化 Q 路径 +2.10 ms/步；
- prefill 的 int8 行构建尚未融合（M4 仅 decode rows≤64）。

## 3. 工程注意

- 实验机上 `csrc/build_aclnn.sh` 曾被替换为 stub，真实脚本保留为 `.real`；
- A3 未实现 `aclrtHostMemMapCapabilities`，用 `HISPARSE_SKIP_HOSTMAP_CAP=1` 跳过；
- gather 按 `HISPARSE_GATHER_CHUNK_BYTES=256 MiB` 分块，避免 4.6–37 GiB 临时张量 OOM。

## 4. 小结

- 内核层的坑多在参数顺序、标量寻址、V/S 栅栏与对齐；
- 剩余优化方向：resolver 下 AI-Core、int8 注意力降差、量化 Q 减负、prefill pack 融合；
- A3 八篇入门结束。对照 `../a5/` 可回看上游基础实现。
